<a href="https://colab.research.google.com/github/jglim531/math-103.1-nlp/blob/main/02_Text_Representation/Sparse_Embeddings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Import Libraries and Models

In [1]:
!pip install scikit-learn

In [2]:
import spacy

nlp = spacy.load("en_core_web_sm")

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer

import pandas as pd

### One-Hot Encoding Example

1. "The cat sat on the mat."
2. "The dog chased the cat!"
3. "Dogs and cats can be friends."
4. "The mat was soft, and the cat liked it."

Pre-processing steps:
- Lowercasing
- Punctuation removal
- Stopword removal (“the”, “on”, “and”, “can”, “be”, “was”, “it”)
- Lemmatization


In [3]:
# Define sentences
sen_1 = "The cat sat on the mat."
sen_2 = "The dog chased the cat!"
sen_3 = "Dogs and cats can be friends."
sen_4 = "The mat was soft, and the cat liked it."

In [4]:
def ohe_preprocessing(sentence):

    # Lowercase the sentence
    sentence = sentence.lower()


    # Remove punctuation
    doc = nlp(sentence)

    cleaned_sentence = []

    for token in doc:
        if token.is_alpha:
            cleaned_sentence.append(token)

    # Join it back to a single sentence
    sentence = " ".join(str(token) for token in cleaned_sentence)


    # Remove stopwords
    doc = nlp(sentence)

    cleaned_sentence = []

    for token in doc:
        if not token.is_stop:
            cleaned_sentence.append(token)

    # Join it back to a single sentence
    sentence = " ".join(str(token) for token in cleaned_sentence)


    # Lemmatize the words
    doc = nlp(sentence)

    cleaned_sentence = []

    for token in doc:
        cleaned_sentence.append(token.lemma_)

    # Join it back to a single sentence
    sentence = " ".join(str(token) for token in cleaned_sentence)

    # Return the final sentence
    return sentence

In [5]:
# Pre-process sentences
pp_sen_1 = ohe_preprocessing(sen_1)
pp_sen_2 = ohe_preprocessing(sen_2)
pp_sen_3 = ohe_preprocessing(sen_3)
pp_sen_4 = ohe_preprocessing(sen_4)

print(f"Sentence 1: {pp_sen_1}")
print(f"Sentence 2: {pp_sen_2}")
print(f"Sentence 3: {pp_sen_3}")
print(f"Sentence 4: {pp_sen_4}")

Sentence 1: cat sit mat
Sentence 2: dog chase cat
Sentence 3: dog cat friend
Sentence 4: mat soft cat like


In [6]:
def collect_vocabulary(list_of_sentences):

    # Define the vocabulary
    vocabulary = []

    for sentence in list_of_sentences:

        # Tokenize the sentence
        doc = nlp(sentence)

        for token in doc:

            # Need to get the text because tokens are specific to the document object
            # A "dog" in sentence 1 is different from a "dog" in sentence 2
            token_text = token.text

            if token_text not in vocabulary:
                vocabulary.append(token_text)
            else:
                continue

    # Sort the vocabulary in alphabetical order
    vocabulary = sorted(vocabulary)

    return vocabulary


In [7]:
# Get vocabulary
ohe_sentences_list = [pp_sen_1, pp_sen_2, pp_sen_3, pp_sen_4]

vocabulary_list = collect_vocabulary(ohe_sentences_list)
print(f"Vocabulary: {vocabulary_list}, Cardinality: {len(vocabulary_list)}")

Vocabulary: ['cat', 'chase', 'dog', 'friend', 'like', 'mat', 'sit', 'soft'], Cardinality: 8


In [8]:
def construct_ohe_vectors(sentence, vocabulary):

    one_hot_encoded_sentence = []

    # Construct the OHE of the word of the given sentence
    for word in sentence.split():

        # Initialize vector
        vector = [0] * len(vocabulary)

        if word in vocabulary:
            index = vocabulary.index(word)
            vector[index] = 1

        one_hot_encoded_sentence.append(vector)

    return one_hot_encoded_sentence

In [9]:
ohe_sen_1 = construct_ohe_vectors(pp_sen_1, vocabulary_list)
ohe_sen_2 = construct_ohe_vectors(pp_sen_2, vocabulary_list)
ohe_sen_3 = construct_ohe_vectors(pp_sen_3, vocabulary_list)
ohe_sen_4 = construct_ohe_vectors(pp_sen_4, vocabulary_list)

In [10]:
# Make the one-hot encoding vectors
print(f"Vocabulary: {vocabulary_list}\n")

preprocessed_sentences = [pp_sen_1, pp_sen_2, pp_sen_3, pp_sen_4]
ohe_sentences = [ohe_sen_1, ohe_sen_2, ohe_sen_3, ohe_sen_4]

for i in range(len(ohe_sentences)):
    print(f"Pre-processed Sentence {i + 1}: {preprocessed_sentences[i]}")
    print(f"OHE Sentence {i + 1}: {ohe_sentences[i]}")
    print("")

Vocabulary: ['cat', 'chase', 'dog', 'friend', 'like', 'mat', 'sit', 'soft']

Pre-processed Sentence 1: cat sit mat
OHE Sentence 1: [[1, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 1, 0], [0, 0, 0, 0, 0, 1, 0, 0]]

Pre-processed Sentence 2: dog chase cat
OHE Sentence 2: [[0, 0, 1, 0, 0, 0, 0, 0], [0, 1, 0, 0, 0, 0, 0, 0], [1, 0, 0, 0, 0, 0, 0, 0]]

Pre-processed Sentence 3: dog cat friend
OHE Sentence 3: [[0, 0, 1, 0, 0, 0, 0, 0], [1, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 1, 0, 0, 0, 0]]

Pre-processed Sentence 4: mat soft cat like
OHE Sentence 4: [[0, 0, 0, 0, 0, 1, 0, 0], [0, 0, 0, 0, 0, 0, 0, 1], [1, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 1, 0, 0, 0]]



### Bag of Words Example

1. "The quick brown fox jumped over the lazy dog."
2. "A clever fox outsmarted the tired dog."
3. "The dog barked at the sneaky fox."
4. "Brown foxes are quick and clever."

Pre-processing steps:
- Lowercasing
- Punctuation removal
- Stopword removal ("the", "a", "over", "at", "are", "and")
- POS Tagging (remove adjectives)
- Lemmatization


In [11]:
# Define sentences
sen_1 = "The quick brown fox jumped over the lazy dog."
sen_2 = "A clever fox outsmarted the tired dog."
sen_3 = "The dog barked at the sneaky fox."
sen_4 = "Brown foxes are quick and clever."

In [12]:
def bow_preprocessing(sentence):

    # Lowercase the sentence
    sentence = sentence.lower()


    # Remove punctuation
    doc = nlp(sentence)

    cleaned_sentence = []

    for token in doc:
        if token.is_alpha:
            cleaned_sentence.append(token)

    # Join it back to a single sentence
    sentence = " ".join(str(token) for token in cleaned_sentence)


    # Remove stopwords
    doc = nlp(sentence)

    cleaned_sentence = []

    for token in doc:
        if not token.is_stop:
            cleaned_sentence.append(token)

    # Join it back to a single sentence
    sentence = " ".join(str(token) for token in cleaned_sentence)


    # Remove the adjectives
    doc = nlp(sentence)

    cleaned_sentence = []

    for token in doc:
        if token.pos_ != "ADJ":
            cleaned_sentence.append(token)

    # Join it back to a single sentence
    sentence = " ".join(str(token) for token in cleaned_sentence)


    # Lemmatize the words
    doc = nlp(sentence)

    cleaned_sentence = []

    for token in doc:
        cleaned_sentence.append(token.lemma_)

    # Join it back to a single sentence
    sentence = " ".join(str(token) for token in cleaned_sentence)

    # Return the final sentence
    return sentence

In [13]:
# Pre-process sentences
pp_sen_1 = bow_preprocessing(sen_1)
pp_sen_2 = bow_preprocessing(sen_2)
pp_sen_3 = bow_preprocessing(sen_3)
pp_sen_4 = bow_preprocessing(sen_4)

print(f"Sentence 1: {pp_sen_1}")
print(f"Sentence 2: {pp_sen_2}")
print(f"Sentence 3: {pp_sen_3}")
print(f"Sentence 4: {pp_sen_4}")

Sentence 1: fox jump dog
Sentence 2: fox outsmart dog
Sentence 3: dog bark fox
Sentence 4: fox


In [14]:
def collect_vocabulary(list_of_sentences):

    # Define the vocabulary
    vocabulary = []

    for sentence in list_of_sentences:

        # Tokenize the sentence
        doc = nlp(sentence)

        for token in doc:

            # Need to get the text because tokens are specific to the document object
            # A "dog" in sentence 1 is different from a "dog" in sentence 2
            token_text = token.text

            if token_text not in vocabulary:
                vocabulary.append(token_text)
            else:
                continue

    # Sort the vocabulary in alphabetical order
    vocabulary = sorted(vocabulary)

    return vocabulary


In [15]:
# Get vocabulary
bow_sentences_list = [pp_sen_1, pp_sen_2, pp_sen_3, pp_sen_4]

vocabulary_list = collect_vocabulary(bow_sentences_list)
print(f"Vocabulary: {vocabulary_list}, Cardinality: {len(vocabulary_list)}")

Vocabulary: ['bark', 'dog', 'fox', 'jump', 'outsmart'], Cardinality: 5


In [16]:
# List of sentences (must be preprocessed already)
bow_sentences_list = [pp_sen_1, pp_sen_2, pp_sen_3, pp_sen_4]

# Create the CountVectorizer (this builds the BoW model)
vectorizer = CountVectorizer()

# Fit and transform the corpus into a BoW matrix
bow_representation = vectorizer.fit_transform(bow_sentences_list)

print(f"Bag of Words Representation:\n")
print(bow_representation)


Bag of Words Representation:

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 10 stored elements and shape (4, 5)>
  Coords	Values
  (0, 2)	1
  (0, 3)	1
  (0, 1)	1
  (1, 2)	1
  (1, 1)	1
  (1, 4)	1
  (2, 2)	1
  (2, 1)	1
  (2, 0)	1
  (3, 2)	1


In [17]:
# Accessing the vector of a specific sentence
print(f"Pre-processed Sentence 1: {pp_sen_1}\n")

bow_sen_1 = bow_representation[0]

# Shows the raw details
print(f"BoW Representation Sentence 1: {bow_sen_1}")
print()

# We need the .toarray() to show the vector representation
bow_sen_1 = bow_sen_1.toarray()
print(f"Vector Representation Sentence 1: {bow_sen_1}")

Pre-processed Sentence 1: fox jump dog

BoW Representation Sentence 1: <Compressed Sparse Row sparse matrix of dtype 'int64'
	with 3 stored elements and shape (1, 5)>
  Coords	Values
  (0, 2)	1
  (0, 3)	1
  (0, 1)	1

Vector Representation Sentence 1: [[0 1 1 1 0]]


In [18]:
bow_sen_1 = bow_representation[0].toarray()
bow_sen_2 = bow_representation[1].toarray()
bow_sen_3 = bow_representation[2].toarray()
bow_sen_4 = bow_representation[3].toarray()

In [19]:
# Get the vocabulary (feature names)
# Can use either .vocabulary_ or .get_feature_names_out()

# Outputs a dictionary of the vocabulary with assigned indices
vocab = vectorizer.vocabulary_
print(f".vocabulary_ output: {vocab}")
print()

# Outputs the raw list of words
vocab = vectorizer.get_feature_names_out()
print(f"get_feature_names_out() output: {vocab}")

.vocabulary_ output: {'fox': 2, 'jump': 3, 'dog': 1, 'outsmart': 4, 'bark': 0}

get_feature_names_out() output: ['bark' 'dog' 'fox' 'jump' 'outsmart']


In [20]:
# Make the bag of words vectors
print(f"Vocabulary: {vectorizer.get_feature_names_out()}\n")

preprocessed_sentences = [pp_sen_1, pp_sen_2, pp_sen_3, pp_sen_4]
bow_sentences = [bow_sen_1, bow_sen_2, bow_sen_3, bow_sen_4]

for i in range(len(ohe_sentences)):
    print(f"Pre-processed Sentence {i + 1}: {preprocessed_sentences[i]}")
    print(f"BOW Sentence {i + 1}: {ohe_sentences[i]}")
    print("")

Vocabulary: ['bark' 'dog' 'fox' 'jump' 'outsmart']

Pre-processed Sentence 1: fox jump dog
BOW Sentence 1: [[1, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 1, 0], [0, 0, 0, 0, 0, 1, 0, 0]]

Pre-processed Sentence 2: fox outsmart dog
BOW Sentence 2: [[0, 0, 1, 0, 0, 0, 0, 0], [0, 1, 0, 0, 0, 0, 0, 0], [1, 0, 0, 0, 0, 0, 0, 0]]

Pre-processed Sentence 3: dog bark fox
BOW Sentence 3: [[0, 0, 1, 0, 0, 0, 0, 0], [1, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 1, 0, 0, 0, 0]]

Pre-processed Sentence 4: fox
BOW Sentence 4: [[0, 0, 0, 0, 0, 1, 0, 0], [0, 0, 0, 0, 0, 0, 0, 1], [1, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 1, 0, 0, 0]]



In [21]:
# Can make a new vector for a sentence containing words from the vocabulary
new_sentence = "dog jump fox, fox outsmart dog, dog bark"

# But, must still follow the same pre-processing
new_sentence = bow_preprocessing(new_sentence)

# Needs to enclose the sentence in a list
new_sentence_representation = vectorizer.transform([new_sentence])

new_sentence_representation = new_sentence_representation.toarray()

print(f"Vocabulary: {vectorizer.get_feature_names_out()}\n")
print(f"New Sentence: {new_sentence}")
print(f"New Sentence Representation: {new_sentence_representation}")


Vocabulary: ['bark' 'dog' 'fox' 'jump' 'outsmart']

New Sentence: dog jump fox fox outsmart dog dog bark
New Sentence Representation: [[1 3 2 1 1]]


In [22]:
# Can make a new vector for a sentence containing words from the vocabulary
new_sentence = "dog jump fox, fox outsmart dog, dog bark, penguin"

# But, must still follow the same pre-processing
new_sentence = bow_preprocessing(new_sentence)

# Needs to enclose the sentence in a list
new_sentence_representation = vectorizer.transform([new_sentence])

new_sentence_representation = new_sentence_representation.toarray()

print(f"Vocabulary: {vectorizer.get_feature_names_out()}\n")
print(f"New Sentence: {new_sentence}")
print(f"New Sentence Representation: {new_sentence_representation}")

print()

# Try for just the one word
just_penguin_sentence = "penguin"
just_penguin_sentence = bow_preprocessing(just_penguin_sentence)
just_penguin_sentence_representation = vectorizer.transform([just_penguin_sentence])
just_penguin_sentence_representation = just_penguin_sentence_representation.toarray()

print(f"Just Penguin Sentence: {just_penguin_sentence}")
print(f"Just Penguin Sentence Representation: {just_penguin_sentence_representation}")

Vocabulary: ['bark' 'dog' 'fox' 'jump' 'outsmart']

New Sentence: dog jump fox fox outsmart dog dog bark penguin
New Sentence Representation: [[1 3 2 1 1]]

Just Penguin Sentence: penguin
Just Penguin Sentence Representation: [[0 0 0 0 0]]


In [23]:
# Compute cosine similarity of sentences with each other

# Loop through all pairs and compute cosine similarity one at a time
for i in range(len(bow_sentences)):
    for j in range(i + 1, len(bow_sentences)):
        # cosine_similarity outputs like this: [[0.66666667]] so need the two [0]'s to access the value itself
        sim = cosine_similarity(bow_sentences[i], bow_sentences[j])[0][0]
        print(f"Cosine similarity between Sentence {i + 1} and Sentence {j + 1}: {sim:.4f}")

print()
print(f"Sentence 1: {pp_sen_1}")
print(f"Sentence 2: {pp_sen_2}")
print(f"Sentence 3: {pp_sen_3}")
print(f"Sentence 4: {pp_sen_4}")

Cosine similarity between Sentence 1 and Sentence 2: 0.6667
Cosine similarity between Sentence 1 and Sentence 3: 0.6667
Cosine similarity between Sentence 1 and Sentence 4: 0.5774
Cosine similarity between Sentence 2 and Sentence 3: 0.6667
Cosine similarity between Sentence 2 and Sentence 4: 0.5774
Cosine similarity between Sentence 3 and Sentence 4: 0.5774

Sentence 1: fox jump dog
Sentence 2: fox outsmart dog
Sentence 3: dog bark fox
Sentence 4: fox


### Bag of N-Grams Example

1. "The penguin swam in the cold ice water."
2. "Penguins love swimming in cold water!"
3. "The emperor penguin is very tall and heavy."
4. "Emperor penguins are tall, heavy birds!"
5. "I bought a new laptop for coding yesterday."

Pre-processing steps:
- Lowercasing
- Punctuation removal
- Stopword removal (“the”, “in”, “is”, “and”, “are”, “for”, “a”, “very”)
- Lemmatization


In [24]:
# Define sentences
sen_1 = "The penguin swam in the cold ice water."
sen_2 = "Penguins love swimming in cold water!"
sen_3 = "The emperor penguin is very tall and heavy."
sen_4 = "Emperor penguins are tall, heavy birds!"
sen_5 = "I bought a new laptop for coding yesterday."


In [25]:
def bon_preprocessing(sentence):

    # Lowercase the sentence
    sentence = sentence.lower()


    # Remove punctuation
    doc = nlp(sentence)

    cleaned_sentence = []

    for token in doc:
        if token.is_alpha:
            cleaned_sentence.append(token)

    # Join it back to a single sentence
    sentence = " ".join(str(token) for token in cleaned_sentence)


    # Remove stopwords (manually since spaCy removes some that are not intended for this exercise)
    manual_stopwords = ["the", "in", "is", "and", "are", "for", "a", "very"]

    cleaned_sentence = []

    words = sentence.split()

    for word in words:
        if word not in manual_stopwords:
            cleaned_sentence.append(word)

    # Join it back to a single sentence
    sentence = " ".join(str(token) for token in cleaned_sentence)


    # Lemmatize the words
    doc = nlp(sentence)

    cleaned_sentence = []

    for token in doc:
        cleaned_sentence.append(token.lemma_)

    # Join it back to a single sentence
    sentence = " ".join(str(token) for token in cleaned_sentence)

    # Return the final sentence
    return sentence

In [26]:
# Pre-process sentences
pp_sen_1 = bon_preprocessing(sen_1)
pp_sen_2 = bon_preprocessing(sen_2)
pp_sen_3 = bon_preprocessing(sen_3)
pp_sen_4 = bon_preprocessing(sen_4)
pp_sen_5 = bon_preprocessing(sen_5)

print(f"Sentence 1: {pp_sen_1}")
print(f"Sentence 2: {pp_sen_2}")
print(f"Sentence 3: {pp_sen_3}")
print(f"Sentence 4: {pp_sen_4}")
print(f"Sentence 5: {pp_sen_5}")

Sentence 1: penguin swam cold ice water
Sentence 2: penguin love swim cold water
Sentence 3: emperor penguin tall heavy
Sentence 4: emperor penguin tall heavy bird
Sentence 5: I buy new laptop coding yesterday


In [27]:
def collect_vocabulary(list_of_sentences):

    # Define the vocabulary
    vocabulary = []

    for sentence in list_of_sentences:

        # Tokenize the sentence
        doc = nlp(sentence)

        for token in doc:

            # Need to get the text because tokens are specific to the document object
            # A "dog" in sentence 1 is different from a "dog" in sentence 2
            token_text = token.text

            if token_text not in vocabulary:
                vocabulary.append(token_text)
            else:
                continue

    # Sort the vocabulary in alphabetical order
    vocabulary = sorted(vocabulary)

    return vocabulary


In [28]:
# Get vocabulary
bon_sentences_list = [pp_sen_1, pp_sen_2, pp_sen_3, pp_sen_4, pp_sen_5]

vocabulary_list = collect_vocabulary(bon_sentences_list)
print(f"Vocabulary: {vocabulary_list}, Cardinality: {len(vocabulary_list)}")

# Delete 'swam' since we lemmatize this to 'swim'
vocabulary_list.remove('swam')
print(f"Vocabulary: {vocabulary_list}, Cardinality: {len(vocabulary_list)}")

Vocabulary: ['I', 'bird', 'buy', 'coding', 'cold', 'emperor', 'heavy', 'ice', 'laptop', 'love', 'new', 'penguin', 'swam', 'swim', 'tall', 'water', 'yesterday'], Cardinality: 17
Vocabulary: ['I', 'bird', 'buy', 'coding', 'cold', 'emperor', 'heavy', 'ice', 'laptop', 'love', 'new', 'penguin', 'swim', 'tall', 'water', 'yesterday'], Cardinality: 16


In [29]:
def bag_of_ngrams(tokenized_sentences, n=1):
    """
    Build a bag-of-n-grams representation using sklearn's CountVectorizer.

    Parameters
    ----------
    tokenized_sentences : list of list of str
        e.g. [pp_sen_1, pp_sen_2, ...] where each pp_sen_i is already tokenized/lemmatized or pre-processed.
    n : int
        n-gram size (1 = unigrams, 2 = bigrams, 3 = trigrams).

    Returns
    -------
    vocabulary : list of str
        The n-gram vocabulary (feature names) learned by the vectorizer.
    matrix : np.ndarray
        Dense count matrix, shape (n_sentences, n_features).
    vectorizer : CountVectorizer
        The fitted vectorizer (in case it is needed later).
    """
    vectorizer = CountVectorizer(
        ngram_range = (n, n),
        lowercase = False,
        token_pattern=r"(?u)\b\w+\b"  # Allow single-character tokens like "i"/"a" since the default needs at least 2 characters
    )

    matrix = vectorizer.fit_transform(tokenized_sentences)
    vocabulary = vectorizer.get_feature_names_out().tolist()

    return vocabulary, matrix.toarray(), vectorizer

In [30]:
def compute_cosine_similarity(matrix, labels=None):
    """
    Compute pairwise cosine similarity across a count matrix using sklearn.

    Parameters
    ----------
    matrix : array-like
        Count matrix, shape (n_sentences, n_features).
    labels : list of str, optional
        Row/column labels for display.

    Returns
    -------
    sim_df : pd.DataFrame
        Pairwise cosine similarity matrix as a labeled DataFrame.
    """
    sim_matrix = cosine_similarity(matrix)

    if labels is None:
        labels = [f"sen_{i+1}" for i in range(matrix.shape[0])]

    sim_df = pd.DataFrame(sim_matrix, index = labels, columns = labels)
    return sim_df

In [31]:
bon_sentences_list = [pp_sen_1, pp_sen_2, pp_sen_3, pp_sen_4, pp_sen_5]
labels = ["sen_1", "sen_2", "sen_3", "sen_4", "sen_5"]

for n in [1, 2, 3]:
    vocab_n, matrix_n, vec_n = bag_of_ngrams(bon_sentences_list, n=n)
    print(f"\n--- n = {n} ---")
    print(f"Vocabulary ({len(vocab_n)}): {vocab_n}")

    sim_df = compute_cosine_similarity(matrix_n, labels=labels)
    print(sim_df.map(lambda x: f"{x:.4f}"))


--- n = 1 ---
Vocabulary (17): ['I', 'bird', 'buy', 'coding', 'cold', 'emperor', 'heavy', 'ice', 'laptop', 'love', 'new', 'penguin', 'swam', 'swim', 'tall', 'water', 'yesterday']
        sen_1   sen_2   sen_3   sen_4   sen_5
sen_1  1.0000  0.6000  0.2236  0.2000  0.0000
sen_2  0.6000  1.0000  0.2236  0.2000  0.0000
sen_3  0.2236  0.2236  1.0000  0.8944  0.0000
sen_4  0.2000  0.2000  0.8944  1.0000  0.0000
sen_5  0.0000  0.0000  0.0000  0.0000  1.0000

--- n = 2 ---
Vocabulary (17): ['I buy', 'buy new', 'coding yesterday', 'cold ice', 'cold water', 'emperor penguin', 'heavy bird', 'ice water', 'laptop coding', 'love swim', 'new laptop', 'penguin love', 'penguin swam', 'penguin tall', 'swam cold', 'swim cold', 'tall heavy']
        sen_1   sen_2   sen_3   sen_4   sen_5
sen_1  1.0000  0.0000  0.0000  0.0000  0.0000
sen_2  0.0000  1.0000  0.0000  0.0000  0.0000
sen_3  0.0000  0.0000  1.0000  0.8660  0.0000
sen_4  0.0000  0.0000  0.8660  1.0000  0.0000
sen_5  0.0000  0.0000  0.0000  0.0000

### TF-IDF Example
1. "The coffee brewed in the morning."
2. "Coffee and tea are hot drinks :)"
3. "The tea was hot, and I liked it."
4. "I brew hot coffee every morning"

Pre-processing steps:
- Lowercasing
- Punctuation removal (keep the happy face)
- Stopword removal (“the”, “in”, “and”, “are”, “was”, “it”)
- Lemmatization


In [32]:
# Define sentences
sen_1 = "The coffee brewed in the morning."
sen_2 = "Coffee and tea are hot drinks :)"
sen_3 = "The tea was hot, and I liked it."
sen_4 = "I brew hot coffee every morning"

In [33]:
def tf_idf_preprocessing(sentence):

    protected_words = {"i", "every", ":)"}

    # Lowercase the sentence
    sentence = sentence.lower()


    # Remove punctuation
    doc = nlp(sentence)

    cleaned_sentence = []

    for token in doc:
        if token.is_alpha or token.text in protected_words:
            cleaned_sentence.append(token)

    # Join it back to a single sentence
    sentence = " ".join(str(token) for token in cleaned_sentence)


    # Remove stopwords
    doc = nlp(sentence)

    cleaned_sentence = []

    for token in doc:
        if not token.is_stop or token.text in protected_words:
            cleaned_sentence.append(token)

    # Join it back to a single sentence
    sentence = " ".join(str(token) for token in cleaned_sentence)


    # Lemmatize the words
    doc = nlp(sentence)

    cleaned_sentence = []

    for token in doc:
        if token.text in protected_words:
            cleaned_sentence.append(token.text) # keep as-is, do not lemmatize
        else:
            cleaned_sentence.append(token.lemma_)

    # Join it back to a single sentence
    sentence = " ".join(str(token) for token in cleaned_sentence)

    # Return the final sentence
    return sentence

In [34]:
# Pre-process sentences
pp_sen_1 = tf_idf_preprocessing(sen_1)
pp_sen_2 = tf_idf_preprocessing(sen_2)
pp_sen_3 = tf_idf_preprocessing(sen_3)
pp_sen_4 = tf_idf_preprocessing(sen_4)

print(f"Sentence 1: {pp_sen_1}")
print(f"Sentence 2: {pp_sen_2}")
print(f"Sentence 3: {pp_sen_3}")
print(f"Sentence 4: {pp_sen_4}")

Sentence 1: coffee brew morning
Sentence 2: coffee tea hot drink :)
Sentence 3: tea hot i like
Sentence 4: i brew hot coffee every morning


In [35]:
def collect_vocabulary(list_of_sentences):

    # Define the vocabulary
    vocabulary = []

    for sentence in list_of_sentences:

        # Tokenize the sentence
        doc = nlp(sentence)

        for token in doc:

            # Need to get the text because tokens are specific to the document object
            # A "dog" in sentence 1 is different from a "dog" in sentence 2
            token_text = token.text

            if token_text not in vocabulary:
                vocabulary.append(token_text)
            else:
                continue

    # Sort the vocabulary in alphabetical order
    vocabulary = sorted(vocabulary)

    return vocabulary

In [36]:
# Get vocabulary
tf_idf_sentences_list = [pp_sen_1, pp_sen_2, pp_sen_3, pp_sen_4]

vocabulary_list = collect_vocabulary(tf_idf_sentences_list)
print(f"Vocabulary: {vocabulary_list}, Cardinality: {len(vocabulary_list)}")

Vocabulary: [':)', 'brew', 'coffee', 'drink', 'every', 'hot', 'i', 'like', 'morning', 'tea'], Cardinality: 10


In [37]:
vectorizer = TfidfVectorizer(vocabulary = vocabulary_list, token_pattern = r"(?u)\b\w+\b|:\)")
tf_idf_matrix = vectorizer.fit_transform(tf_idf_sentences_list)

print(f"TF-IDF matrix shape: {tf_idf_matrix.shape}")
print(tf_idf_matrix.toarray())

cosine_sim_matrix = cosine_similarity(tf_idf_matrix)
print(f"\nCosine similarity matrix:\n{cosine_sim_matrix}")

TF-IDF matrix shape: (4, 10)
[[0.         0.61366674 0.49681612 0.         0.         0.
  0.         0.         0.61366674 0.        ]
 [0.53944516 0.         0.34432086 0.53944516 0.         0.34432086
  0.         0.         0.         0.42530476]
 [0.         0.         0.         0.         0.         0.39205255
  0.4842629  0.61422608 0.         0.4842629 ]
 [0.         0.41101031 0.33274827 0.         0.52131446 0.33274827
  0.41101031 0.         0.41101031 0.        ]]

Cosine similarity matrix:
[[1.         0.17106415 0.         0.66976142]
 [0.17106415 1.         0.34095119 0.22914434]
 [0.         0.34095119 1.         0.32949185]
 [0.66976142 0.22914434 0.32949185 1.        ]]


In [38]:
# As dense arrays
print(f"Vocabulary: {vocabulary_list}")
print()

tf_idf_array = tf_idf_matrix.toarray()

for i, sentence in enumerate(tf_idf_sentences_list):
    vector = tf_idf_array[i]
    print(f"Sentence {i+1}: {sentence}")
    print(f"TF-IDF vector: {vector}\n")

Vocabulary: [':)', 'brew', 'coffee', 'drink', 'every', 'hot', 'i', 'like', 'morning', 'tea']

Sentence 1: coffee brew morning
TF-IDF vector: [0.         0.61366674 0.49681612 0.         0.         0.
 0.         0.         0.61366674 0.        ]

Sentence 2: coffee tea hot drink :)
TF-IDF vector: [0.53944516 0.         0.34432086 0.53944516 0.         0.34432086
 0.         0.         0.         0.42530476]

Sentence 3: tea hot i like
TF-IDF vector: [0.         0.         0.         0.         0.         0.39205255
 0.4842629  0.61422608 0.         0.4842629 ]

Sentence 4: i brew hot coffee every morning
TF-IDF vector: [0.         0.41101031 0.33274827 0.         0.52131446 0.33274827
 0.41101031 0.         0.41101031 0.        ]



In [39]:
import pandas as pd

labels = [f"Sentence {i + 1}" for i in range(len(tf_idf_sentences_list))]
cosine_sim_df = pd.DataFrame(cosine_sim_matrix, index = labels, columns = labels)
print(cosine_sim_df)

            Sentence 1  Sentence 2  Sentence 3  Sentence 4
Sentence 1    1.000000    0.171064    0.000000    0.669761
Sentence 2    0.171064    1.000000    0.340951    0.229144
Sentence 3    0.000000    0.340951    1.000000    0.329492
Sentence 4    0.669761    0.229144    0.329492    1.000000
